# Multi-output model

[Notebook](custom-multi-output.ipynb)

Wrap one PyTorch model that returns several scores per sequence. Increase `target` and decrease `off_target` with twice the weight on `target`.

## Define the model



In [ ]:
# %pip install "bioseq-models @ git+https://github.com/UTR-CODE/bioseq-models.git@84a5075c3656127d99027c38705212a2de6e62c9"

import torch
from torch import nn
from MotifSift import MultiTaskObjective, SequencePredictor, generate_designs, random_seed_sequences


class ExampleModel(nn.Module):
    def __init__(self, patterns):
        super().__init__()
        self.conv = nn.Conv1d(4, len(patterns), kernel_size=3)
        with torch.no_grad():
            self.conv.weight.zero_()
            self.conv.bias.fill_(-2)
            for channel, pattern in enumerate(patterns):
                for position, base in enumerate(pattern):
                    self.conv.weight[channel, "ACGT".index(base), position] = 1

    def forward(self, x):
        return self.conv(x).relu().sum(dim=-1)




## Connect the model and generate candidates



In [2]:
model = SequencePredictor(ExampleModel(["ACG", "GTA"]), tasks=["target", "off_target"])
objective = MultiTaskObjective(
    high=["target"], low=["off_target"], weights={"target": 2},
)

length = 24
seeds = random_seed_sequences(length, 8)
designs = generate_designs(
    model, length=length, initial=seeds, objective=objective,
    num_candidates=8, score_threshold=1,
)
designs.groupby("algorithm")["score"].agg(["size", "median", "max"])


,size,median,max
algorithm,,,
adalead,8,0.000000,1.000000
basesearch,8,0.666667,1.000000
beam_search,8,0.000000,1.000000
cross_entropy_method,8,1.333333,1.333333
den,8,0.666667,1.333333
fast_seqprop,8,0.666667,1.333333
genetic_algorithm,8,0.000000,1.333333
ism_greedy,8,0.666667,1.000000
ledidi,8,0.000000,0.333333
